# Notebook 43 — Scale Inflection Test

**Does d_min plateau once a window captures enough cycles of the dominant process — and does this plateau fail for multi-process signals?**

---

## Background

nb42 (F125–F127) confirmed the dominant-process hypothesis: Spearman ρ=0.932 between process coherence score and d_min. The tidal gauge produced d=0.724 — the cleanest classification in the corpus — because gravitational forcing is the single dominant process at all temporal scales tested (1-week, 1-month, 1-year).

But nb42 only tested three fixed scales. A finer question: **does d_min depend on window length at all for a single-process signal?** And conversely, does a multi-process signal show window-length sensitivity that a single-process signal does not?

**The mechanism:** A pure sinusoidal process has one characteristic timescale — its period. Once a window captures several complete cycles, the fingerprint (skewness, kurtosis, lag1, ZC, slope, baseline_delta) stabilises. Adding more cycles adds no new information. The window-length curve of d_min should plateau above the cycle-capture threshold.

For a multi-process signal, each process has its own timescale. At short windows, the fastest process dominates the fingerprint; at longer windows, slower processes emerge. The fingerprint is never fully stable — d_min remains variable across scales.

---

## Design

**Signal A — Tidal gauge (single dominant process):**
The M2 semi-diurnal tide has period 12.42h. Scan window lengths from 6h (sub-cycle) to 8760h (full year). Expect: d_min large below 12.5h, drops sharply once the window captures ≥1 M2 cycle, then plateaus.

**Signal B — Intel Lab thermistor hourly (competing processes):**
The dominant cycle is diurnal (24h from solar + HVAC), competing with multi-day building occupancy patterns and weather intrusions. Scan window lengths from 6h to 480h (~20 days, the data extent). Expect: d_min more variable — no stable plateau.

**Signal C — Scale stability comparison:**
For each signal, compute the coefficient of variation of d_min across all window lengths above the nominal cycle length. Single-process signals should have lower CV (more stable classification).

---

## Pre-run predictions

**F128:** Tidal d_min drops to < 1.5 once window ≥ 12.5h (one M2 cycle) and stays below 1.5 for all longer windows. The plateau is flat: CV(d_min, windows ≥ 24h) < 0.15.

**F129:** Thermistor d_min does not plateau. It varies more across scale — CV(d_min, windows ≥ 24h) > 0.30. The scale scan reveals fingerprint instability from competing HVAC/occupancy processes.

**F130:** The sub-cycle boundary is detectable as a sharp d_min inflection near the dominant cycle period (12.42h for tidal, ~24h for thermistor). Below the cycle threshold, both signals look like trend/integrated_trend fragments. The inflection point is a measurable property of the dominant process period.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import gzip, io, sys
sys.path.insert(0, '..')
from data_utils import get_dataset

SIGNED_COLS = ['skewness', 'kurtosis', 'lag1_autocorr', 'zero_crossings', 'slope', 'baseline_delta']
SEQ_LEN = 64; SEED = 42; t64 = np.linspace(0, 1, SEQ_LEN)

def zscore(s):
    s = np.asarray(s, dtype=float); std = s.std()
    return (s - s.mean()) / std if std > 1e-8 else s - s.mean()

def baseline_delta_fn(s, frac=0.10):
    k = max(1, int(len(s) * frac))
    return float(np.mean(s[-k:]) - np.mean(s[:k]))

def extract_6f(s):
    arr = np.asarray(s, dtype=float); t = np.arange(len(arr))
    lag1 = float(np.corrcoef(arr[:-1], arr[1:])[0, 1]) if len(arr) > 2 else 0.0
    return {
        'skewness':       float(stats.skew(arr)),
        'kurtosis':       float(stats.kurtosis(arr)),
        'lag1_autocorr':  lag1,
        'zero_crossings': float(np.sum(np.diff(np.sign(arr)) != 0) / len(arr)),
        'slope':          float(stats.linregress(t, arr).slope),
        'baseline_delta': baseline_delta_fn(arr),
    }

GENERATORS = {
    'burst':              lambda r: zscore(np.exp(-(t64-r.uniform(.15,.50))**2/(2*r.uniform(.05,.15)**2))+r.normal(0,.05,SEQ_LEN)),
    'oscillator':         lambda r: zscore(np.sin(2*np.pi*r.uniform(1.5,4.5)*t64+r.uniform(0,np.pi))+r.normal(0,.05,SEQ_LEN)),
    'seasonal':           lambda r: zscore(np.sin(2*np.pi*r.uniform(3,6)*t64)+.25*np.sin(4*np.pi*r.uniform(3,6)*t64)+r.normal(0,.04,SEQ_LEN)),
    'trend':              lambda r: zscore(t64+r.uniform(.05,.30)*t64**2+r.normal(0,.02,SEQ_LEN)),
    'integrated_trend':   lambda r: zscore(np.cumsum(np.ones(SEQ_LEN)*r.uniform(.015,.035)+r.normal(0,.003,SEQ_LEN))),
    'irregular_osc':      lambda r: zscore((np.sin(2*np.pi*r.uniform(2,5)*t64)*(1+r.uniform(.3,.8,SEQ_LEN))+r.normal(0,.3,SEQ_LEN))*1.4),
    'declining_osc':      lambda r: zscore(np.linspace(r.uniform(.9,1.2),r.uniform(.35,.65),SEQ_LEN)*np.sin(2*np.pi*r.uniform(2.5,5.5)*t64)+np.linspace(0,r.uniform(-.8,-.4),SEQ_LEN)+r.normal(0,.05,SEQ_LEN)),
    'declining_monotonic':lambda r: zscore(np.cumsum(-np.ones(SEQ_LEN)*r.uniform(.015,.035)+r.normal(0,.003,SEQ_LEN))),
}

recs = []
for cls, gen in GENERATORS.items():
    for i in range(200):
        r = np.random.default_rng(SEED + list(GENERATORS).index(cls)*1000 + i)
        f = extract_6f(gen(r)); f['class'] = cls; recs.append(f)
df_train = pd.DataFrame(recs)
sc = StandardScaler()
X_tr = sc.fit_transform(df_train[SIGNED_COLS].values)
ctrds_8 = {c: X_tr[df_train['class']==c].mean(axis=0) for c in GENERATORS}

def classify(feat_dict, ctrds=ctrds_8):
    x = sc.transform([[feat_dict[c] for c in SIGNED_COLS]])[0]
    dists = {c: float(np.linalg.norm(x - v)) for c, v in ctrds.items()}
    return min(dists, key=dists.get), dists

print('8-class centroid classifier ready.')

In [ ]:
# ---- Load cached signals from nb42 ----

# Tidal gauge (The Battery, NYC, 2023) — already local from nb42
raw_tide = get_dataset('noaa_battery_tidal_2023.csv', lambda: b'')
df_tide  = pd.read_csv(io.BytesIO(raw_tide))
wl_col   = next((c for c in df_tide.columns if 'Water Level' in c or 'water' in c.lower()), df_tide.columns[1])
tide_raw = pd.to_numeric(df_tide[wl_col], errors='coerce').dropna().values
print(f'Tidal gauge: {len(tide_raw):,} hourly values')

# Intel Lab thermistor (moteid 48, nb41+42) — already local
raw_intel = get_dataset('intel_lab_sensors.txt.gz', lambda: b'')
with gzip.open(io.BytesIO(raw_intel)) as f:
    text = f.read().decode('utf-8', errors='ignore')

rows = []
for line in text.split('
'):
    parts = line.strip().split()
    if len(parts) >= 7:
        try:
            rows.append({'epoch': int(parts[2]), 'moteid': int(parts[3]), 'temperature': float(parts[4])})
        except:
            pass

df_intel   = pd.DataFrame(rows)
best_moteid = (
    df_intel[(df_intel['temperature'] > 15) & (df_intel['temperature'] < 40)]
    .groupby('moteid').size().idxmax()
)
s_best     = df_intel[df_intel['moteid'] == best_moteid].sort_values('epoch')
s_best     = s_best[(s_best['temperature'] > 15) & (s_best['temperature'] < 40)]
therm_full = s_best['temperature'].values

rph        = int(3600 / 31)           # ~116 readings per hour
n_hours    = len(therm_full) // rph
therm_hrly = np.array([therm_full[i*rph:(i+1)*rph].mean() for i in range(n_hours)])
# Start from hour 0 (include full range for longer windows)
print(f'Thermistor hourly: {n_hours} hourly means, range [{therm_hrly.min():.1f}, {therm_hrly.max():.1f}] °C')
print(f'Max window: {n_hours}h = {n_hours/24:.1f} days')

In [ ]:
# ---- Part A: Scale scan — Tidal gauge ----
# Scan 22 log-spaced window lengths from 6h to 8760h.
# At each length, take the first n points (anchored at start of year).
# Record: class, d_min, dominant 6-feature fingerprint.
# M2 period = 12.42h → expect inflection near there.

import io  # ensure io in scope

M2_PERIOD = 12.42  # hours

tide_windows_h = np.unique(np.round(
    np.logspace(np.log10(6), np.log10(8760), 24)
).astype(int))
tide_windows_h = [w for w in tide_windows_h if w <= len(tide_raw)]

print('Tidal gauge scale scan (window lengths in hours):')
print(f'  Windows: {tide_windows_h}')
print()

tide_results = []
print(f'  {"window_h":>8s}  {"n_cycles":>8s}  {"class":>20s}  {"d_min":>6s}')
for wh in tide_windows_h:
    seg = zscore(tide_raw[:wh])
    fp  = extract_6f(seg)
    cls, dists = classify(fp)
    dm  = dists[cls]
    n_cycles = wh / M2_PERIOD
    tide_results.append({'window_h': wh, 'n_cycles_M2': n_cycles, 'class': cls, 'd_min': dm, **fp})
    print(f'  {wh:>8d}  {n_cycles:>8.2f}  {cls:>20s}  {dm:>6.3f}')

df_tide_scan = pd.DataFrame(tide_results)

# CV of d_min for windows >= 24h (multi-cycle regime)
multi_cycle_tide = df_tide_scan[df_tide_scan['window_h'] >= 24]['d_min']
cv_tide = multi_cycle_tide.std() / multi_cycle_tide.mean()
print(f'
  CV(d_min, windows >= 24h) = {cv_tide:.4f}')
print(f'  Plateau (CV < 0.15): {cv_tide < 0.15}')

# Inflection: index of steepest drop in d_min
diffs = np.diff(df_tide_scan['d_min'].values)
inflection_idx = np.argmin(diffs)  # biggest negative step
inflection_h   = df_tide_scan['window_h'].values[inflection_idx]
print(f'  Steepest d_min drop at window = {inflection_h}h  (M2 period = {M2_PERIOD}h)')

In [ ]:
# ---- Part B: Scale scan — Thermistor hourly ----
# Scan window lengths from 6h to max available (~480h / 20 days).
# Diurnal period ~24h → expect inflection near there.
# Prediction: d_min does not plateau; CV > 0.30.

DIURNAL_PERIOD = 24.0  # hours (dominant solar + HVAC cycle)

max_therm_h = len(therm_hrly)
therm_windows_h = np.unique(np.round(
    np.logspace(np.log10(6), np.log10(max_therm_h), 22)
).astype(int))
therm_windows_h = [w for w in therm_windows_h if w <= max_therm_h]

print('Thermistor hourly scale scan:')
print(f'  Windows: {therm_windows_h}')
print()

therm_results = []
print(f'  {"window_h":>8s}  {"n_cycles":>8s}  {"class":>20s}  {"d_min":>6s}')
for wh in therm_windows_h:
    seg = zscore(therm_hrly[:wh])
    fp  = extract_6f(seg)
    cls, dists = classify(fp)
    dm  = dists[cls]
    n_cycles = wh / DIURNAL_PERIOD
    therm_results.append({'window_h': wh, 'n_cycles_diurnal': n_cycles, 'class': cls, 'd_min': dm, **fp})
    print(f'  {wh:>8d}  {n_cycles:>8.2f}  {cls:>20s}  {dm:>6.3f}')

df_therm_scan = pd.DataFrame(therm_results)

multi_cycle_therm = df_therm_scan[df_therm_scan['window_h'] >= 24]['d_min']
cv_therm = multi_cycle_therm.std() / multi_cycle_therm.mean()
print(f'
  CV(d_min, windows >= 24h) = {cv_therm:.4f}')
print(f'  No plateau (CV > 0.30): {cv_therm > 0.30}')

diffs_t = np.diff(df_therm_scan['d_min'].values)
inflection_idx_t = np.argmin(diffs_t)
inflection_h_t   = df_therm_scan['window_h'].values[inflection_idx_t]
print(f'  Steepest d_min drop at window = {inflection_h_t}h  (diurnal period = {DIURNAL_PERIOD}h)')

In [ ]:
# ---- Part C: Scale stability comparison ----
# Compare CV values across windows >= dominant cycle length.
# Also measure: class entropy (how many distinct classes appear across scales).
# Single-process: CV low, class entropy low (same class at all scales).
# Multi-process: CV high, class entropy higher.

from collections import Counter

def scale_stability(df_scan, min_window_h, cycle_col='d_min'):
    sub = df_scan[df_scan['window_h'] >= min_window_h]
    dm  = sub['d_min'].values
    cv  = dm.std() / dm.mean() if dm.mean() > 0 else np.nan
    cls_counts = Counter(sub['class'].values)
    n_distinct  = len(cls_counts)
    dominant_cls = cls_counts.most_common(1)[0][0]
    dominant_frac = cls_counts.most_common(1)[0][1] / len(sub)
    return {'cv': cv, 'n_distinct_classes': n_distinct,
            'dominant_class': dominant_cls, 'dominant_frac': dominant_frac,
            'd_min_mean': dm.mean(), 'd_min_std': dm.std(),
            'd_min_min': dm.min(), 'd_min_max': dm.max()}

stab_tide  = scale_stability(df_tide_scan,  min_window_h=24)
stab_therm = scale_stability(df_therm_scan, min_window_h=24)

print('Scale stability summary (windows >= 24h):')
print(f'  {"Metric":35s}  {"Tidal":>10s}  {"Thermistor":>10s}')
print('  ' + '-'*60)
for k in ['cv', 'n_distinct_classes', 'dominant_class', 'dominant_frac',
          'd_min_mean', 'd_min_std', 'd_min_min', 'd_min_max']:
    vt = stab_tide[k]; vh = stab_therm[k]
    if isinstance(vt, float):
        print(f'  {k:35s}  {vt:>10.3f}  {vh:>10.3f}')
    else:
        print(f'  {k:35s}  {str(vt):>10s}  {str(vh):>10s}')

print()
print(f'  Tidal CV < 0.15 (F128): {stab_tide["cv"] < 0.15}')
print(f'  Thermistor CV > 0.30 (F129): {stab_therm["cv"] > 0.30}')
print()

# Class transitions across scale
print('Tidal class sequence across scales:')
for _, row in df_tide_scan[['window_h', 'class', 'd_min']].iterrows():
    print(f'  {row["window_h"]:>6}h  →  {row["class"]:>20s}  d={row["d_min"]:.3f}')
print()
print('Thermistor class sequence across scales:')
for _, row in df_therm_scan[['window_h', 'class', 'd_min']].iterrows():
    print(f'  {row["window_h"]:>6}h  →  {row["class"]:>20s}  d={row["d_min"]:.3f}')

In [ ]:
# ---- Visualization ----
CLASS_COLORS = {
    'oscillator': '#2196F3', 'declining_osc': '#9C27B0',
    'burst': '#F44336', 'seasonal': '#FF9800', 'trend': '#795548',
    'integrated_trend': '#607D8B', 'irregular_osc': '#E91E63',
    'declining_monotonic': '#009688',
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel A: d_min vs window length — tidal
ax = axes[0, 0]
x_t = df_tide_scan['window_h'].values
y_t = df_tide_scan['d_min'].values
cls_t = df_tide_scan['class'].values
for i, (x, y, c) in enumerate(zip(x_t, y_t, cls_t)):
    ax.scatter(x, y, color=CLASS_COLORS.get(c, 'gray'), s=60, zorder=3)
ax.plot(x_t, y_t, color='#2196F3', lw=1.5, alpha=0.6)
ax.axvline(M2_PERIOD, color='red', ls='--', lw=1.5, label=f'M2 period ({M2_PERIOD}h)')
ax.axhline(1.5, color='gray', ls=':', lw=1, label='d=1.5 threshold')
ax.set_xscale('log')
ax.set_xlabel('Window length (hours, log scale)')
ax.set_ylabel('d_min')
ax.set_title(f'Tidal gauge: d_min vs window length
CV(d_min, ≥24h) = {stab_tide["cv"]:.3f}', fontsize=10)
ax.legend(fontsize=8)

# Panel B: d_min vs window length — thermistor
ax = axes[0, 1]
x_h = df_therm_scan['window_h'].values
y_h = df_therm_scan['d_min'].values
cls_h = df_therm_scan['class'].values
for i, (x, y, c) in enumerate(zip(x_h, y_h, cls_h)):
    ax.scatter(x, y, color=CLASS_COLORS.get(c, 'gray'), s=60, zorder=3)
ax.plot(x_h, y_h, color='#E91E63', lw=1.5, alpha=0.6)
ax.axvline(DIURNAL_PERIOD, color='orange', ls='--', lw=1.5, label=f'Diurnal period ({DIURNAL_PERIOD}h)')
ax.axhline(1.5, color='gray', ls=':', lw=1, label='d=1.5 threshold')
ax.set_xscale('log')
ax.set_xlabel('Window length (hours, log scale)')
ax.set_ylabel('d_min')
ax.set_title(f'Thermistor hourly: d_min vs window length
CV(d_min, ≥24h) = {stab_therm["cv"]:.3f}', fontsize=10)
ax.legend(fontsize=8)

# Panel C: Overlay both curves (normalised to same axis, aligned at 24h)
ax = axes[1, 0]
ax.plot(x_t, y_t, color='#2196F3', lw=2, label=f'Tidal (CV={stab_tide["cv"]:.3f})', marker='o', ms=5)
# thermistor on same axis (x-axis in hours, y-axis d_min)
ax.plot(x_h, y_h, color='#E91E63', lw=2, label=f'Thermistor (CV={stab_therm["cv"]:.3f})', marker='s', ms=5)
ax.axvline(M2_PERIOD, color='#2196F3', ls=':', lw=1, alpha=0.5)
ax.axvline(DIURNAL_PERIOD, color='#E91E63', ls=':', lw=1, alpha=0.5)
ax.set_xscale('log')
ax.set_xlabel('Window length (hours, log scale)')
ax.set_ylabel('d_min')
ax.set_title('Tidal vs thermistor: scale stability comparison', fontsize=10)
ax.legend(fontsize=9)

# Panel D: Class map — colour-coded class vs window length for both signals
ax = axes[1, 1]
# tidal row at y=1, thermistor row at y=0
for _, row in df_tide_scan.iterrows():
    ax.barh(1.1, 1, left=np.log10(row['window_h']), height=0.4,
            color=CLASS_COLORS.get(row['class'], 'gray'), alpha=0.8)
for _, row in df_therm_scan.iterrows():
    ax.barh(0.1, 1, left=np.log10(row['window_h']), height=0.4,
            color=CLASS_COLORS.get(row['class'], 'gray'), alpha=0.8)
ax.set_yticks([0.1, 1.1])
ax.set_yticklabels(['Thermistor', 'Tidal'])
ax.set_xlabel('log10(window hours)')
ax.set_title('Class assignment by scale (colour = class)', fontsize=10)
from matplotlib.patches import Patch
legend_els = [Patch(facecolor=v, label=k) for k, v in CLASS_COLORS.items()]
ax.legend(handles=legend_els, fontsize=6, loc='upper left', ncol=2)

fig.suptitle('Notebook 43 — Scale Inflection Test
d_min vs window length: plateau (single-process) vs variance (multi-process)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('43_scale_inflection.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

---
## Findings — Notebook 43

### F128 — [Tidal scale scan — to be filled after nb43 run]

**Prediction:** d_min < 1.5 for all windows ≥ 12.5h (one M2 cycle); CV(d_min, ≥24h) < 0.15.

**Result:** [TBD]

**What it means:** [TBD]

---

### F129 — [Thermistor scale scan — to be filled after nb43 run]

**Prediction:** d_min does not plateau; CV(d_min, ≥24h) > 0.30.

**Result:** [TBD]

**What it means:** [TBD]

---

### F130 — [Scale inflection point — to be filled after nb43 run]

**Prediction:** The steepest d_min drop for tidal is near 12.5h (M2 period); for thermistor near 24h (diurnal period).

**Result:** [TBD]

**What it means:** [TBD]

---

Findings F128–F130 added. Total findings: **[TBD]**.